# Testing for models
Models :
- OLS Linear Regression 
- AR
- (ARMA)
- VAR
- (HAR)

Variables :
- Baseline : Stock price = f(oil price)
- Macro-state interaction : Stock price = f(oil price, D_macro_state) (CFNAI)
- Macro-state + shock type : Stock price = f(oil price, D_macro_state, D_shock_type)

CFNAI is monthly so our period reference will be monthly. 
Daily data will be aggregated to monthly by taking the last observation of the month.

In [20]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

## OLS regression

In [21]:

def ols(X,y):
    '''
    OLS regression : Stock price = f(oil price)
    '''
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()

    return model.summary()

### Baseline

In [22]:
# Data preparation
daily_data = pd.read_excel("../data/raw/data_hec_project_1.xlsx", skiprows=5, sheet_name="Daily")
monthly_data = pd.read_excel("../data/raw/data_hec_project_1.xlsx", skiprows=5, sheet_name="Monthly")

daily_data.columns = daily_data.columns.astype(str).str.strip()
date_col = "Date" if "Date" in daily_data.columns else "Dates"
daily_data[date_col] = pd.to_datetime(daily_data[date_col], errors="coerce")
daily_data = daily_data.dropna(subset=[date_col]).set_index(date_col).sort_index()

SP = daily_data["SP500"].replace(["#N/A N/A", "NA", ""], pd.NA).apply(pd.to_numeric, errors="coerce")
WTI = daily_data["WTI"].replace(["#N/A N/A", "NA", ""], pd.NA).apply(pd.to_numeric, errors="coerce")

# Daily log-returns
log_returns_SP = np.log(SP).diff().dropna()
log_returns_WTI = np.log(WTI).diff().dropna()

# Predictive regression: Y_t = f(r_oil_{t-1})
df_daily_pred = pd.concat([
    log_returns_SP.rename("Y"),
    log_returns_WTI.rename("r_oil").shift(1),
], axis=1).dropna()
print(ols(df_daily_pred[["r_oil"]], df_daily_pred["Y"]))


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     10.30
Date:                Tue, 31 Mar 2026   Prob (F-statistic):            0.00134
Time:                        09:38:01   Log-Likelihood:                 29043.
No. Observations:                9441   AIC:                        -5.808e+04
Df Residuals:                    9439   BIC:                        -5.807e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0003      0.000      2.726      0.0

### Macro-state interaction

In [ ]:
# Monthly log-returns: resample to end-of-month prices first, then log-diff per series
# (dropna must come AFTER diff — dropping prices before diff would create multi-month gaps)
log_returns_SP_monthly  = np.log(SP .resample("ME").last()).diff().dropna()
log_returns_WTI_monthly = np.log(WTI.resample("ME").last()).diff().dropna()

# CFNAI
md = monthly_data.copy()
md.columns = md.columns.astype(str).str.strip()
md["Dates"] = pd.to_datetime(md["Dates"], errors="coerce")
md = md.dropna(subset=["Dates"])
md["Dates"] = md["Dates"].dt.to_period("M").dt.to_timestamp("M")
md = md.set_index("Dates").sort_index()

CFNAI = pd.to_numeric(md["CFNAI"].replace(["#N/A N/A", "NA", ""], pd.NA), errors="coerce")
D_exp = (CFNAI > 0).astype(int).rename("D_exp")
D_con = (CFNAI < -0.7).astype(int).rename("D_con")

# Build df_reg with contemporaneous data (used by downstream cells)
df_reg = pd.concat([
    log_returns_SP_monthly.rename("Y"),
    log_returns_WTI_monthly.rename("r_oil"),
    D_exp,
    D_con,
], axis=1).dropna()
df_reg["r_oil_x_D_exp"] = df_reg["r_oil"] * df_reg["D_exp"]
df_reg["r_oil_x_D_con"] = df_reg["r_oil"] * df_reg["D_con"]

# Predictive regression: Y_t = f(X_{t-1})
X_cols_reg = ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"]
df_reg_pred = pd.concat([df_reg["Y"], df_reg[X_cols_reg].shift(1)], axis=1).dropna()
X = df_reg_pred[X_cols_reg]
y = df_reg_pred["Y"]

print(ols(X, y))


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.034
Model:                            OLS   Adj. R-squared:                  0.023
Method:                 Least Squares   F-statistic:                     3.006
Date:                Tue, 31 Mar 2026   Prob (F-statistic):             0.0111
Time:                        09:38:01   Log-Likelihood:                 758.23
No. Observations:                 432   AIC:                            -1504.
Df Residuals:                     426   BIC:                            -1480.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0077      0.003      2.387

C:\Users\danny\AppData\Local\Temp\ipykernel_9496\404655992.py:19: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_reg = pd.concat([


### Macro state + type of shocks

In [24]:
df3 = df_reg.copy()

# High yield and US treasury yields
HY   = daily_data["HY"].replace(["#N/A N/A", "NA", ""], pd.NA).apply(pd.to_numeric, errors="coerce")
US10 = daily_data["US10"].replace(["#N/A N/A", "NA", ""], pd.NA).apply(pd.to_numeric, errors="coerce")
US2  = daily_data["US2"].replace(["#N/A N/A", "NA", ""], pd.NA).apply(pd.to_numeric, errors="coerce")

HY_monthly   = HY.resample("ME").last()
US10_monthly = US10.resample("ME").last()
US2_monthly  = US2.resample("ME").last()

# CreditSpread and Slope
CreditSpread = (HY_monthly - US10_monthly).rename("CreditSpread")
Slope        = (US10_monthly - US2_monthly).rename("Slope")

df3 = df3.join(pd.concat([CreditSpread, Slope], axis=1), how="left")

# Load shock data
sh = pd.read_excel("../data/external/shock_types.xlsx")
sh.columns = sh.columns.astype(str).str.strip()
sh["Start"]      = pd.to_datetime(sh["Start"], errors="coerce")
sh["End"]        = pd.to_datetime(sh["End"], errors="coerce")
sh["Shock_type"] = pd.to_numeric(sh["Shock_type"], errors="coerce")
sh = sh.dropna(subset=["Start", "End", "Shock_type"]).sort_values(["Start", "End"])

# Monthly shock type series (0 = no shock, 1 = supply, 2 = demand, 3 = geo)
month_start = df3.index.to_period("M").to_timestamp("M")
shock_type = pd.Series(0, index=df3.index, name="Shock_type")
for _, row in sh.iterrows():
    mask = (month_start >= row["Start"]) & (month_start <= row["End"])
    shock_type.loc[mask] = int(row["Shock_type"])

# D_sup for supply, D_dem for demand, D_geo for geopolitical
shock_dummies = pd.get_dummies(shock_type, prefix="D_shock", dtype=int)
shock_dummies = shock_dummies.drop(columns=["D_shock_0"] if "D_shock_0" in shock_dummies.columns else [])
shock_dummies = shock_dummies.rename(columns={"D_shock_1": "D_sup", "D_shock_2": "D_dem", "D_shock_3": "D_geo"})

df3 = df3.join(shock_dummies, how="left")
df3[shock_dummies.columns] = df3[shock_dummies.columns].fillna(0)

# r_oil * shock interactions (contemporaneous, used for downstream df_model/df_opt)
shock_cols = list(shock_dummies.columns)
for c in shock_cols:
    df3[f"r_oil_x_{c}"] = df3["r_oil"] * df3[c]

X3_cols = (
    ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"]
    + shock_cols
    + [f"r_oil_x_{c}" for c in shock_cols]
    + ["CreditSpread", "Slope"]
)

# Predictive regression: Y_t = f(X_{t-1})
df3_pred = pd.concat([df3["Y"], df3[X3_cols].shift(1)], axis=1).dropna()
X3 = df3_pred[X3_cols]
y3 = df3_pred["Y"]

print(ols(X3, y3))


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.050
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     1.697
Date:                Tue, 31 Mar 2026   Prob (F-statistic):             0.0588
Time:                        09:38:01   Log-Likelihood:                 761.85
No. Observations:                 432   AIC:                            -1496.
Df Residuals:                     418   BIC:                            -1439.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0163      0.007      2.453

#### Results (predictive: Y_t = f(X_{t-1})) :
Significant :
- r_oil *** : lagged oil predicts SP500 in a neutral macro regime (D_exp=D_con=0)
- r_oil × D_exp *** : oil's predictive effect is significantly amplified during expansions
- r_oil × D_con ** : oil's predictive effect differs significantly during contractions

Not significant :
- D_exp, D_con : regime states carry no independent level effect once lagged
- D_sup, D_dem, D_geo + all interactions : oil shock origin irrelevant for forecasting
- CreditSpread, Slope : financial conditions at t-1 carry no 1-month-ahead predictive power

R² = 0.05, low but non-trivial for monthly equity returns.
=> Oil does have predictive content, but only through macro-regime interactions. Without controlling for the macro state, the oil signal is not useful. This model is to be compared with the extended controls below (where msciem is added).

## AR Model

### Optimizing AR : solving for p 

In [25]:
p_max = 12
base = pd.concat([log_returns_SP_monthly.rename("Y"), log_returns_WTI_monthly.rename("r_oil")], axis=1)
rows = []
for p in range(1, p_max + 1):
    d = base.copy()
    d["r_oil_lag1"] = d["r_oil"].shift(1)   # r_oil_{t-1}: known at forecast time
    for lag in range(1, p + 1):
        d[f"Y_lag{lag}"] = d["Y"].shift(lag)
    d = d.dropna()
    X_p = sm.add_constant(d[[f"Y_lag{lag}" for lag in range(1, p + 1)] + ["r_oil_lag1"]])
    m = sm.OLS(d["Y"], X_p).fit()
    rows.append({"p": p, "AIC": round(m.aic, 2), "BIC": round(m.bic, 2)})

res = pd.DataFrame(rows).set_index("p")
print(res.to_string())
print(f"\nBest p by AIC: {res['AIC'].idxmin()}")
print(f"Best p by BIC: {res['BIC'].idxmin()}")


        AIC      BIC
p                   
1  -1502.74 -1490.53
2  -1496.82 -1480.54
3  -1492.76 -1472.43
4  -1490.08 -1465.70
5  -1484.15 -1455.72
6  -1480.35 -1447.88
7  -1482.54 -1446.02
8  -1479.39 -1438.84
9  -1473.51 -1428.94
10 -1469.24 -1420.65
11 -1462.84 -1410.23
12 -1458.03 -1401.40

Best p by AIC: 1
Best p by BIC: 1


### Baseline

In [26]:
df_ar1 = pd.concat([
    log_returns_SP_monthly.rename("Y"),
    log_returns_WTI_monthly.rename("r_oil"),
], axis=1)
df_ar1["Y_lag1"]    = df_ar1["Y"].shift(1)
df_ar1["r_oil_lag1"] = df_ar1["r_oil"].shift(1)
df_ar1 = df_ar1[["Y", "Y_lag1", "r_oil_lag1"]].dropna()

# Predictive regression: Y_t = f(Y_{t-1}, r_oil_{t-1})
X_ar1 = df_ar1[["Y_lag1", "r_oil_lag1"]]
y_ar1 = df_ar1["Y"]

print(ols(X_ar1, y_ar1))


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     1.508
Date:                Tue, 31 Mar 2026   Prob (F-statistic):              0.223
Time:                        09:38:01   Log-Likelihood:                 754.37
No. Observations:                 433   AIC:                            -1503.
Df Residuals:                     430   BIC:                            -1491.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0069      0.002      3.337      0.0

### Macro state

In [27]:
X_ar2_cols = ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"]

df_ar2 = df_reg[["Y"] + X_ar2_cols].copy()
df_ar2["Y_lag1"] = df_ar2["Y"].shift(1)
# Lag all external predictors to t-1
for col in X_ar2_cols:
    df_ar2[col] = df_ar2[col].shift(1)
df_ar2 = df_ar2.dropna()

# Predictive regression: Y_t = f(Y_{t-1}, X_{t-1})
X_ar2 = df_ar2[["Y_lag1"] + X_ar2_cols]
y_ar2 = df_ar2["Y"]

print(ols(X_ar2, y_ar2))


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.034
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     2.527
Date:                Tue, 31 Mar 2026   Prob (F-statistic):             0.0205
Time:                        09:38:01   Log-Likelihood:                 758.31
No. Observations:                 432   AIC:                            -1503.
Df Residuals:                     425   BIC:                            -1474.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0077      0.003      2.361

### Macro state + shock types

In [28]:
shock_cols_ar     = [c for c in df3.columns if c in ["D_sup", "D_dem", "D_geo"]]
shock_int_cols_ar = [f"r_oil_x_{c}" for c in shock_cols_ar if f"r_oil_x_{c}" in df3.columns]

X_ar3_ext = (
    ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"]
    + shock_cols_ar + shock_int_cols_ar
    + ["CreditSpread", "Slope"]
)

df_ar3 = df3[["Y"] + X_ar3_ext].copy()
df_ar3["Y_lag1"] = df_ar3["Y"].shift(1)
# Lag all external predictors to t-1
for col in X_ar3_ext:
    df_ar3[col] = df_ar3[col].shift(1)

X_ar3_cols = ["Y_lag1"] + X_ar3_ext
X_ar3 = df_ar3[X_ar3_cols].dropna()
y_ar3 = df_ar3.loc[X_ar3.index, "Y"]

# Predictive regression: Y_t = f(Y_{t-1}, X_{t-1})
print(ols(X_ar3, y_ar3))

model = sm.OLS(y_ar3, sm.add_constant(X_ar3)).fit()

from statsmodels.stats.diagnostic import acorr_ljungbox
acorr_ljungbox(model.resid, lags=[5], return_df=True)


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.050
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     1.580
Date:                Tue, 31 Mar 2026   Prob (F-statistic):             0.0815
Time:                        09:38:01   Log-Likelihood:                 761.91
No. Observations:                 432   AIC:                            -1494.
Df Residuals:                     417   BIC:                            -1433.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0167      0.007      2.464

,lb_stat,lb_pvalue
5,3.707773,0.592208


#### Results :
- Y_lag1: not expected to add much, consistent with VAR result that SP500 returns have low autocorrelation
- Adding AR dynamics to the shock-type OLS (which showed R² = 0.05) brings limited improvement
- r_oil and macro-regime interactions *, *** retain predictive content as in the OLS version
- Ljung-Box: check residual autocorrelation — AR term redundant if Y is not autoregressive

## VAR Model
### Baseline

In [29]:
from statsmodels.tsa.api import VAR

var_data = pd.concat([
    log_returns_SP_monthly.rename("Y"),
    log_returns_WTI_monthly.rename("r_oil"),
], axis=1).dropna()

# Select lag order by AIC, max 12
var_model = VAR(var_data)
lag_sel = var_model.select_order(maxlags=12)
print(lag_sel.summary())

p_var = lag_sel.aic
var_result = var_model.fit(p_var)
print(var_result.summary())

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -10.96     -10.94*   1.743e-05     -10.95*
1       -10.97      -10.92   1.717e-05      -10.95
2      -10.98*      -10.88  1.706e-05*      -10.94
3       -10.97      -10.83   1.728e-05      -10.91
4       -10.98      -10.81   1.708e-05      -10.91
5       -10.97      -10.76   1.726e-05      -10.88
6       -10.96      -10.71   1.742e-05      -10.86
7       -10.95      -10.66   1.755e-05      -10.84
8       -10.94      -10.62   1.771e-05      -10.81
9       -10.93      -10.56   1.800e-05      -10.78
10      -10.91      -10.51   1.822e-05      -10.75
11      -10.91      -10.47   1.830e-05      -10.73
12      -10.91      -10.44   1.820e-05      -10.73
--------------------------------------------------
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 31, Mar

#### Results
**From the Y equation** (what predicts SP500) :
- L1.r_oil p = 0.063 : oil has borderline predictive power **in a bivariate model** (not significant at 5%)
- L2.r_oil p = 0.142 : second lag clearly not significant
=> Weak Granger causality from oil to SP500 in the bivariate VAR. Important: macro-regime interactions (expansion/contraction) are absent here, which is why OLS with interactions finds stronger signal.

**From the r_oil equation** (what predicts oil) :
- L1.r_oil p = 0.002, L2.r_oil p = 0.034 : oil IS autoregressive — useful for oil forecasting
- L1.Y p = 0.572, L2.Y p = 0.354 : SP500 does NOT Granger-cause oil

Corr(Y, oil) = 0.225 → contemporaneous link exists, driven by common shocks within the same month.
=> The VAR result is consistent with later OLS findings: oil's predictive power requires regime context (macro state interactions). A bivariate VAR without regime dummies underestimates this effect.

### Testing for optimal regression

Only significative variables

In [30]:
cols = ["Y", "r_oil", "D_con", "D_geo", "CreditSpread"]

# Build df_model with contemporaneous data (used downstream by df_opt)
df_model = df3[cols].copy()
df_model["r_oil_x_D_geo"] = df_model["r_oil"] * df_model["D_geo"]
df_model = df_model.dropna()

# Predictive regression: Y_t = f(X_{t-1})
X_cols_model = ["r_oil", "D_con", "D_geo", "r_oil_x_D_geo", "CreditSpread"]
df_model_pred = pd.concat([df_model["Y"], df_model[X_cols_model].shift(1)], axis=1).dropna()

X = df_model_pred[X_cols_model]
y = df_model_pred["Y"]

X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     1.795
Date:                Tue, 31 Mar 2026   Prob (F-statistic):              0.113
Time:                        09:38:01   Log-Likelihood:                 755.25
No. Observations:                 432   AIC:                            -1498.
Df Residuals:                     426   BIC:                            -1474.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0142      0.005      2.696

Treat all dummies (Macro state and shocks) as only neutral or not

In [31]:
cols = ["Y", "r_oil", "D_exp", "D_con", "D_sup", "D_dem", "D_geo", "CreditSpread"]

df_model2 = df3[cols].copy()
df_model2["D_macro"] = ((df_model2["D_exp"] == 1) | (df_model2["D_con"] == 1)).astype(int)
df_model2["D_shock"] = (
    (df_model2["D_sup"] == 1)
    | (df_model2["D_dem"] == 1)
    | (df_model2["D_geo"] == 1)
).astype(int)
df_model2["r_oil_x_D_shock"] = df_model2["r_oil"] * df_model2["D_shock"]

use_cols = ["Y", "r_oil", "D_macro", "D_shock", "r_oil_x_D_shock", "CreditSpread"]
df_model2 = df_model2[use_cols].dropna()

# Predictive regression: Y_t = f(X_{t-1})
X_cols_m2 = ["r_oil", "D_macro", "D_shock", "r_oil_x_D_shock", "CreditSpread"]
df_m2_pred = pd.concat([df_model2["Y"], df_model2[X_cols_m2].shift(1)], axis=1).dropna()

X2 = df_m2_pred[X_cols_m2]
y2 = df_m2_pred["Y"]

model2 = sm.OLS(y2, sm.add_constant(X2)).fit()
print(model2.summary())


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.017
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     1.445
Date:                Tue, 31 Mar 2026   Prob (F-statistic):              0.207
Time:                        09:38:01   Log-Likelihood:                 754.38
No. Observations:                 432   AIC:                            -1497.
Df Residuals:                     426   BIC:                            -1472.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.0133      0.006     

Tester 
y = wSP * eSP + wB * eB + Wc * eB
=> Conscequences dans la vraie vie, le oil fait allouer y a Sp ou B

tester les trois y separement

### Adding other controls

In [ ]:
# Clean daily price series for NA strings before computing end-of-month log-returns
def _clean(col): return daily_data[col].replace(["#N/A N/A","NA",""], pd.NA).apply(pd.to_numeric, errors="coerce")

r_gold   = np.log(_clean("XAU")  .resample("ME").last()).diff().rename("r_gold")
r_natgas = np.log(_clean("NatGas").resample("ME").last()).diff().rename("r_natgas")
r_msciem = np.log(_clean("MSCIEM").resample("ME").last()).diff().rename("r_msciem")
d_US10   = US10_monthly.diff().rename("d_US10")   # reuse already-cleaned monthly US10 from above

IP_m     = pd.to_numeric(md["IP"]    .replace(["#N/A N/A","NA",""], pd.NA), errors="coerce")
MISM_m   = pd.to_numeric(md["MISM"]  .replace(["#N/A N/A","NA",""], pd.NA), errors="coerce")
MISMPP_m = pd.to_numeric(md["MISMPP"].replace(["#N/A N/A","NA",""], pd.NA), errors="coerce")
USR_m    = pd.to_numeric(md["USR"]   .replace(["#N/A N/A","NA",""], pd.NA), errors="coerce")
RFI_m    = pd.to_numeric(md["RFI"]   .replace(["#N/A N/A","NA",""], pd.NA), errors="coerce")

d_log_IP  = np.log(IP_m).diff().rename("d_log_IP")
ISM_mfg   = (MISM_m   - 50).rename("ISM_mfg")
ISM_pp    = (MISMPP_m  - 50).rename("ISM_pp")
d_log_USR = np.log(USR_m).diff().rename("d_log_USR")
RFI       = RFI_m.rename("RFI")

new_controls = pd.concat(
    [r_gold, r_natgas, r_msciem, d_US10, d_log_IP, ISM_mfg, ISM_pp, d_log_USR, RFI],
    axis=1, sort=True
)
df_ext = df_model.join(new_controls, how="left").dropna()

X_ext_cols = [
    "r_oil", "D_con", "D_geo", "r_oil_x_D_geo", "CreditSpread",
    "r_gold", "r_natgas", "r_msciem", "d_US10",
    "d_log_IP", "ISM_mfg", "ISM_pp", "d_log_USR", "RFI"
]

# Predictive regression: Y_t = f(X_{t-1})

df_ext_pred = pd.concat([df_ext["Y"], df_ext[X_ext_cols].shift(1)], axis=1).dropna()
print(ols(df_ext_pred[X_ext_cols], df_ext_pred["Y"]))

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.053
Model:                            OLS   Adj. R-squared:                  0.016
Method:                 Least Squares   F-statistic:                     1.432
Date:                Tue, 31 Mar 2026   Prob (F-statistic):              0.143
Time:                        09:38:01   Log-Likelihood:                 597.84
No. Observations:                 348   AIC:                            -1168.
Df Residuals:                     334   BIC:                            -1114.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0114      0.009      1.336

Variables selected to reduce multicollinearity from the extended model :
- r_msciem (global equity market proxy)
- r_gold
- d_log_USR (unemployment rate change — slow-moving real economy signal)
- ISM_mfg (manufacturing sentiment)

Note : selection was based on contemporaneous in-sample fit. In the predictive framework, return-based variables (r_msciem, r_gold) are not expected to retain significance — only d_log_USR is a genuine leading indicator.

In [33]:
# Build df_opt with contemporaneous data (used downstream by FWL and test cells)
df_opt = df_model.join(
    pd.concat([r_msciem, r_gold, d_log_USR, ISM_mfg], axis=1, sort=True),
    how="left"
).dropna()

X_opt_cols = ["r_oil", "D_con", "D_geo", "r_oil_x_D_geo", "CreditSpread",
              "r_msciem", "r_gold", "d_log_USR", "ISM_mfg"]

# Predictive regression: Y_t = f(X_{t-1})
df_opt_pred = pd.concat([df_opt["Y"], df_opt[X_opt_cols].shift(1)], axis=1).dropna()
print(ols(df_opt_pred[X_opt_cols], df_opt_pred["Y"]))


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.038
Model:                            OLS   Adj. R-squared:                  0.013
Method:                 Least Squares   F-statistic:                     1.500
Date:                Tue, 31 Mar 2026   Prob (F-statistic):              0.146
Time:                        09:38:01   Log-Likelihood:                 595.22
No. Observations:                 348   AIC:                            -1170.
Df Residuals:                     338   BIC:                            -1132.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0102      0.008      1.233

#### Results (predictive) :
- Low predictive fit : R² = 0.038
- d_log_USR ** : the only consistent predictor — lagged unemployment change signals future equity weakness
- D_geo * : geopolitical shock state has marginal predictive value (persistent regime)
- r_oil : NOT significant — no predictive power once lagged
- r_msciem, r_gold : NOT significant — return series carry no forecasting information at 1-month lag
- CreditSpread, ISM_mfg : NOT significant — financial conditions don't improve forecast at this horizon

Note : the original "good fit" and "msciem/creditspread drive the market" conclusions were based on a contemporaneous regression, not a forecast. Here, only the real economy signal (unemployment) is predictive.

In [34]:
X_no_dummies = ["r_oil", "CreditSpread", "r_msciem", "r_gold", "d_log_USR"]

# Predictive regression: Y_t = f(X_{t-1})
df_nd_pred = pd.concat([df_opt["Y"], df_opt[X_no_dummies].shift(1)], axis=1).dropna()
print(ols(df_nd_pred[X_no_dummies], df_nd_pred["Y"]))


                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.013
Method:                 Least Squares   F-statistic:                     1.936
Date:                Tue, 31 Mar 2026   Prob (F-statistic):             0.0878
Time:                        09:38:01   Log-Likelihood:                 593.26
No. Observations:                 348   AIC:                            -1175.
Df Residuals:                     342   BIC:                            -1151.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0126      0.006      2.245   

Oil not significant with or without the other controls in the predictive framework. Only d_log_USR ** retains predictive power.
Economically : oil prices are absorbed into equity prices within the same month. No exploitable signal remains one month ahead.
Oil does not "capture" the forecasting content of msciem or CreditSpread — those variables are also not predictive. The earlier interpretation was wrong : oil was absorbing contemporaneous co-movement, not genuine leading information.

## Frisch–Waugh–Lovell (FWL) Test
Does `r_oil` have an **independent** effect on `Y` after controlling for macro-financial variables?

In [35]:
controls = ["r_msciem", "CreditSpread", "r_gold", "d_log_USR"]

# ── 1. Data preparation: predictive setup Y_t ~ X_{t-1} ──────────────────────
df_fwl_raw = df_opt[["Y", "r_oil"] + controls]
df_fwl = pd.concat([df_fwl_raw["Y"], df_fwl_raw[["r_oil"] + controls].shift(1)], axis=1).dropna()

Y     = df_fwl["Y"]
X_oil = df_fwl["r_oil"]
Z     = sm.add_constant(df_fwl[controls])

# ── 2. Residualise Y on controls_{t-1} ───────────────────────────────────────
Y_resid = sm.OLS(Y, Z).fit().resid

# ── 3. Residualise r_oil_{t-1} on controls_{t-1} ─────────────────────────────
X_resid = sm.OLS(X_oil, Z).fit().resid.rename("r_oil")

# ── 4. FWL regression: Y_resid ~ X_resid ─────────────────────────────────────
fwl_model = sm.OLS(Y_resid, sm.add_constant(X_resid)).fit()
print("── FWL regression: Y_resid ~ r_oil_resid (predictive) ──────────────────")
print(fwl_model.summary())

# ── 5. Validation: full predictive model ─────────────────────────────────────
full_model = sm.OLS(Y, sm.add_constant(df_fwl[["r_oil"] + controls])).fit()
print("\n── Full model: Y_t ~ r_oil_{t-1} + controls_{t-1} ──────────────────────")
print(full_model.summary())

# ── 6. Coefficient match check ────────────────────────────────────────────────
fwl_coef  = fwl_model.params["r_oil"]
full_coef = full_model.params["r_oil"]
print(f"\nFWL  coefficient on r_oil : {fwl_coef:.6f}")
print(f"Full coefficient on r_oil : {full_coef:.6f}")
print(f"Coefficients match        : {abs(fwl_coef - full_coef) < 1e-8}")


── FWL regression: Y_resid ~ r_oil_resid (predictive) ──────────────────
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     2.508
Date:                Tue, 31 Mar 2026   Prob (F-statistic):              0.114
Time:                        09:38:01   Log-Likelihood:                 593.26
No. Observations:                 348   AIC:                            -1183.
Df Residuals:                     346   BIC:                            -1175.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------

In [ ]:
other_controls = ["CreditSpread", "r_gold", "d_log_USR"]

df_fwl2_raw = df_opt[["Y", "r_oil", "r_msciem"] + other_controls]
df_fwl2 = pd.concat(
    [df_fwl2_raw["Y"], df_fwl2_raw[["r_oil", "r_msciem"] + other_controls].shift(1)],
    axis=1
).dropna()

Y2 = df_fwl2["Y"]
Z2 = sm.add_constant(df_fwl2[other_controls])

Y_resid2       = sm.OLS(Y2,                  Z2).fit().resid
r_oil_resid    = sm.OLS(df_fwl2["r_oil"],    Z2).fit().resid.rename("r_oil")
r_msciem_resid = sm.OLS(df_fwl2["r_msciem"], Z2).fit().resid.rename("r_msciem")

X_resid2 = pd.concat([r_oil_resid, r_msciem_resid], axis=1)

fwl_model2 = sm.OLS(Y_resid2, sm.add_constant(X_resid2)).fit()
print("── FWL: Y_t ~ r_oil_{t-1} + r_msciem_{t-1}  (controls: CreditSpread, r_gold, d_log_USR, all at t-1) ──")
print(fwl_model2.summary())


── FWL: Y_t ~ r_oil_{t-1} + r_msciem_{t-1}  (controls: CreditSpread, r_gold, d_log_USR, all at t-1) ──
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     1.251
Date:                Tue, 31 Mar 2026   Prob (F-statistic):              0.288
Time:                        09:38:01   Log-Likelihood:                 593.26
No. Observations:                 348   AIC:                            -1181.
Df Residuals:                     345   BIC:                            -1169.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------

In the predictive FWL (all at t-1) :
- FWL full model : only d_log_USR ** remains significant (R² = 0.028). r_oil and r_msciem are both not significant.
- FWL joint (residualising r_oil and r_msciem on CreditSpread, r_gold, d_log_USR) : R² = 0.007, NOTHING significant.

=> Neither lagged oil nor lagged global equity returns carry any independent forecasting power. The oil–msciem contemporaneous correlation reflects simultaneous reaction to global shocks within the same month — not a predictive relationship. The search for conditions where oil adds extra info continues below.

In [37]:
BASE_MODEL = ["r_oil", "r_msciem", "CreditSpread", "r_gold", "d_log_USR"]

# Pull missing columns from df3 (D_sup, r_oil_x_D_sup, r_oil_x_D_con)
extra = df3[["D_sup", "r_oil_x_D_sup", "r_oil_x_D_con"]].copy()
df_test = df_opt.join(extra, how="left").dropna()

TEST_MODEL1 = BASE_MODEL + ["D_sup", "r_oil_x_D_sup"]
TEST_MODEL2 = BASE_MODEL + ["D_con", "r_oil_x_D_con"]

# Predictive regression: Y_t = f(X_{t-1})
df_t1_pred = pd.concat([df_test["Y"], df_test[TEST_MODEL1].shift(1)], axis=1).dropna()
print("── TEST MODEL 1: BASE + Supply shock dummy + interaction ───────────────")
print(ols(df_t1_pred[TEST_MODEL1], df_t1_pred["Y"]))

df_t2_pred = pd.concat([df_test["Y"], df_test[TEST_MODEL2].shift(1)], axis=1).dropna()
print("── TEST MODEL 2: BASE + Contraction dummy + interaction ────────────────")
print(ols(df_t2_pred[TEST_MODEL2], df_t2_pred["Y"]))


── TEST MODEL 1: BASE + Supply shock dummy + interaction ───────────────
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     1.412
Date:                Tue, 31 Mar 2026   Prob (F-statistic):              0.199
Time:                        09:38:01   Log-Likelihood:                 593.39
No. Observations:                 348   AIC:                            -1171.
Df Residuals:                     340   BIC:                            -1140.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------

In [38]:
threshold = df_test["r_oil"].abs().quantile(0.9)

df_test = df_test.copy()
df_test["D_big"] = (df_test["r_oil"].abs() > threshold).astype(int)
df_test["r_oil_x_big"] = df_test["r_oil"] * df_test["D_big"]

TEST_MODEL3 = BASE_MODEL + ["r_oil_x_big"]

print(f"Threshold: ±{threshold:.4f}  |  D_big={df_test['D_big'].sum()} obs")

# Predictive regression: Y_t = f(X_{t-1})
df_t3_pred = pd.concat([df_test["Y"], df_test[TEST_MODEL3].shift(1)], axis=1).dropna()
print("── TEST MODEL 3: BASE + large oil shock interaction ────────────────────")
print(ols(df_t3_pred[TEST_MODEL3], df_t3_pred["Y"]))


Threshold: ±0.1504  |  D_big=35 obs
── TEST MODEL 3: BASE + large oil shock interaction ────────────────────
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     1.609
Date:                Tue, 31 Mar 2026   Prob (F-statistic):              0.144
Time:                        09:38:01   Log-Likelihood:                 593.26
No. Observations:                 348   AIC:                            -1173.
Df Residuals:                     341   BIC:                            -1146.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------

#### Summary of conditional tests :
- TEST MODEL 1 (supply shock) : r_oil * marginally significant with D_sup included, but r_oil x D_sup NOT significant -> oil effect does not specifically amplify during supply shocks
- TEST MODEL 2 (contraction) : D_con and r_oil x D_con both NOT significant -> recession regime does not reveal extra oil predictive content
- TEST MODEL 3 (large oil moves) : r_oil x big NOT significant -> even extreme oil price changes carry no residual predictive signal

In all three tests : d_log_USR is the only stable predictor; R² ≈ 0.028 unchanged.

Conclusion : oil and msciem correlate contemporaneously because both react to global shocks in the same month. Neither is a useful 1-month-ahead leading indicator. The only variable that reliably forecasts SP500 returns is the lagged change in unemployment, a slow-moving real economy signal. This is consistent with efficient market theory: financial prices are absorbed immediately, only real activity measures carry predictable forward-looking information.